# Credit Risk Model Development

Trains and evaluates the `HistGradientBoostingClassifier` served by this
repository's API (`app/main.py`, `src/credit_risk/`). This notebook is the
full, leakage-checked pipeline that produced every number reported in the
project README and `model_artifacts/model_config.json`.

**Dataset:** Kaggle ["Give Me Some Credit"](https://www.kaggle.com/c/GiveMeSomeCredit)
(`cs-training.csv`) — 150,000 applicants, ~6.7% positive (serious
delinquency within two years).

**How to run:** download `cs-training.csv` from Kaggle, place it at
`data/cs-training.csv` in the repo root (see `data/README.md`), then run
this notebook with its working directory set to `training/` (the default
when opened directly from that folder).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

## 1. Load the data

In [ ]:
DATA_PATH = Path("../data/cs-training.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Download 'cs-training.csv' from the Kaggle 'Give Me Some Credit' "
        "competition and place it at data/cs-training.csv in the repo root "
        "(see data/README.md)."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape}")


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Columns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

## 2. Target distribution and class imbalance

In [ ]:
target_col = "SeriousDlqin2yrs"

print("Target distribution:")
print(df[target_col].value_counts())

print("\nTarget distribution (%):")
print(
    df[target_col]
    .value_counts(normalize=True)
    .mul(100)
    .round(3)
)

default_rate = df[target_col].mean()

print(f"\nDefault rate: {default_rate:.4%}")

## 3. Data quality audit — missingness, duplicates, anomalous values

The special delinquency codes `96`/`98` and extreme utilization values found here are *not* deleted — they're handled explicitly in the feature-preparation steps below instead.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing = missing.sort_values(
    "missing_percent",
    ascending=False
)

missing

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

if "Unnamed: 0" in df.columns:
    print(f"Duplicate IDs: {df['Unnamed: 0'].duplicated().sum()}")

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

suspicious_summary = pd.DataFrame({
    "min": df[numeric_columns].min(),
    "max": df[numeric_columns].max(),
    "mean": df[numeric_columns].mean(),
    "median": df[numeric_columns].median(),
    "zeros": (df[numeric_columns] == 0).sum(),
    "missing": df[numeric_columns].isna().sum()
})

suspicious_summary

In [ ]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

for col in delinquency_cols:
    print(df[col].value_counts().sort_index().tail(15))

for col in delinquency_cols:
    count_98 = (df[col] == 98).sum()
    default_rate_98 = df.loc[df[col] == 98, target_col].mean()

    print(
        f"{col}: "
        f"{count_98} records with value 98, "
        f"default rate = {default_rate_98:.2%}"
    )

In [ ]:
util_col = "RevolvingUtilizationOfUnsecuredLines"

print("Utilization > 1:")
print((df[util_col] > 1).sum())

print("\nUtilization > 10:")
print((df[util_col] > 10).sum())

print("\nLargest utilization values:")
print(df[util_col].nlargest(20).to_list())

In [ ]:
age_col = "age"

print("Age statistics:")
print(df[age_col].describe())

print(f"\nAge <= 0: {(df[age_col] <= 0).sum()}")
print(f"Age < 18: {(df[age_col] < 18).sum()}")
print(f"Age > 100: {(df[age_col] > 100).sum()}")

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x=target_col
)

plt.title("Target Distribution")
plt.xlabel("Serious Delinquency Within 2 Years")
plt.ylabel("Number of Applicants")

plt.show()

In [ ]:
audit_summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_monthly_income_pct": round(
        df["MonthlyIncome"].isna().mean() * 100, 3
    ),
    "missing_dependents_pct": round(
        df["NumberOfDependents"].isna().mean() * 100, 3
    ),
    "default_count": int(df[target_col].sum()),
    "non_default_count": int((df[target_col] == 0).sum()),
    "default_rate_pct": round(df[target_col].mean() * 100, 3)
}

audit_summary

## 4. Separate ID, target, and the audit-only attribute

`age` is split off into `audit_df` here and never joins the feature matrix `X` below. It's carried through the train/test split so it's available for the fairness audit in Section 16, without ever being visible to `.fit()`.

In [ ]:
# Separate the dataset identifier from modeling data.
ID_COL = "Unnamed: 0"
TARGET_COL = "SeriousDlqin2yrs"

applicant_id = df[ID_COL].copy()

df_model = df.drop(columns=[ID_COL]).copy()

print("Original shape:", df.shape)
print("Modeling dataframe shape:", df_model.shape)
print("ID column removed from modeling data:", ID_COL not in df_model.columns)

In [ ]:
AUDIT_COLS = [
    "age"
]

audit_df = df_model[AUDIT_COLS].copy()

print("Audit-only columns:")
print(AUDIT_COLS)

In [ ]:
y = df_model[TARGET_COL].copy()

print("Target:", TARGET_COL)
print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

In [ ]:
MODEL_COLS = [
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

X = df_model[MODEL_COLS].copy()

print("Number of model features:", len(MODEL_COLS))
print("\nModel features:")
for feature in MODEL_COLS:
    print("-", feature)

In [ ]:
# Safety checks: prevent accidental leakage.

assert TARGET_COL not in X.columns, \
    "Target column leaked into model features!"

assert ID_COL not in X.columns, \
    "ID column leaked into model features!"

assert not set(AUDIT_COLS).intersection(X.columns), \
    "Audit-only attribute leaked into model features!"

assert len(X) == len(y) == len(audit_df), \
    "X, y and audit data have different numbers of rows!"

print("All feature leakage checks passed.")

## 5. Train/test split — stratified, with an explicit leakage check

The split is stratified on the target and uses a fixed seed. The next cell independently verifies there's no row overlap between the train and test applicant IDs.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, audit_train, audit_test = train_test_split(
    X,
    y,
    audit_df,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

In [ ]:
print("TRAIN")
print("-" * 40)
print("Rows:", len(X_train))
print("Defaults:", int(y_train.sum()))
print("Default rate:", f"{y_train.mean():.4%}")

print("\nTEST")
print("-" * 40)
print("Rows:", len(X_test))
print("Defaults:", int(y_test.sum()))
print("Default rate:", f"{y_test.mean():.4%}")

print(
    "Train/Test sizes:",
    X_train.shape,
    X_test.shape
)

In [ ]:
train_ids, test_ids = train_test_split(
    applicant_id,
    test_size=0.20,
    random_state=42,
    stratify=y
)

overlap = set(train_ids).intersection(set(test_ids))

print("Train IDs:", len(train_ids))
print("Test IDs:", len(test_ids))
print("Overlapping IDs:", len(overlap))

assert len(overlap) == 0, "Data leakage: applicant appears in both train and test!"

print("No applicant overlap detected.")

## 6. Baseline model: class-weighted logistic regression

Preprocessing (imputation + scaling) lives inside the `Pipeline`, and `cross_validate` refits it on each fold's training data only — this is what makes the CV scores leakage-free.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

print("Step 3 libraries imported successfully.")

In [ ]:
NUMERIC_FEATURES = MODEL_COLS.copy()

assert set(NUMERIC_FEATURES) == set(X_train.columns)
assert len(NUMERIC_FEATURES) == 9

print("Numeric features:")
for feature in NUMERIC_FEATURES:
    print("-", feature)

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES
        )
    ],
    remainder="drop"
)

print(preprocessor)

In [ ]:
baseline_classifier = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", baseline_classifier)
    ]
)

print(baseline_pipeline)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

In [ ]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "neg_brier": "neg_brier_score"
}

print("CV metrics:", list(scoring.keys()))

In [ ]:
cv_results = cross_validate(
    estimator=baseline_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("Cross-validation completed.")

In [ ]:
cv_results_df = pd.DataFrame({
    "Fold": range(1, 6),
    "ROC_AUC": cv_results["test_roc_auc"],
    "PR_AUC": cv_results["test_pr_auc"],
    "Brier_Score": -cv_results["test_neg_brier"]
})

cv_results_df

In [ ]:
cv_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score"
    ],
    "Mean": [
        cv_results_df["ROC_AUC"].mean(),
        cv_results_df["PR_AUC"].mean(),
        cv_results_df["Brier_Score"].mean()
    ],
    "Std": [
        cv_results_df["ROC_AUC"].std(),
        cv_results_df["PR_AUC"].std(),
        cv_results_df["Brier_Score"].std()
    ]
})

cv_summary

In [ ]:
print(
    f"ROC-AUC: "
    f"{cv_results_df['ROC_AUC'].mean():.4f} "
    f"± {cv_results_df['ROC_AUC'].std():.4f}"
)

print(
    f"PR-AUC:  "
    f"{cv_results_df['PR_AUC'].mean():.4f} "
    f"± {cv_results_df['PR_AUC'].std():.4f}"
)

print(
    f"Brier:   "
    f"{cv_results_df['Brier_Score'].mean():.4f} "
    f"± {cv_results_df['Brier_Score'].std():.4f}"
)

In [ ]:
print("Pipeline steps:")
for name, step in baseline_pipeline.named_steps.items():
    print(f"{name}: {type(step).__name__}")

print("\nPreprocessing steps:")
for name, step in numeric_pipeline.named_steps.items():
    print(f"{name}: {type(step).__name__}")

## 7. Why class weighting, not SMOTE — a measured ablation

Rather than asserting that class weighting is better, this compares the weighted and unweighted variants directly on identical folds.

In [ ]:
unweighted_classifier = LogisticRegression(
    class_weight=None,
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

unweighted_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", unweighted_classifier)
    ]
)

print("Unweighted logistic regression pipeline created.")

In [ ]:
unweighted_cv_results = cross_validate(
    estimator=unweighted_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("Unweighted cross-validation completed.")

In [ ]:
weighted_metrics = {
    "Model": "Logistic Regression (class-weighted)",
    "ROC-AUC Mean": cv_results["test_roc_auc"].mean(),
    "ROC-AUC Std": cv_results["test_roc_auc"].std(),
    "PR-AUC Mean": cv_results["test_pr_auc"].mean(),
    "PR-AUC Std": cv_results["test_pr_auc"].std(),
    "Brier Mean": -cv_results["test_neg_brier"].mean(),
    "Brier Std": (-cv_results["test_neg_brier"]).std()
}

unweighted_metrics = {
    "Model": "Logistic Regression (unweighted)",
    "ROC-AUC Mean": unweighted_cv_results["test_roc_auc"].mean(),
    "ROC-AUC Std": unweighted_cv_results["test_roc_auc"].std(),
    "PR-AUC Mean": unweighted_cv_results["test_pr_auc"].mean(),
    "PR-AUC Std": unweighted_cv_results["test_pr_auc"].std(),
    "Brier Mean": -unweighted_cv_results["test_neg_brier"].mean(),
    "Brier Std": (-unweighted_cv_results["test_neg_brier"]).std()
}

imbalance_comparison = pd.DataFrame(
    [weighted_metrics, unweighted_metrics]
)

imbalance_comparison

In [ ]:
roc_auc_difference = (
    weighted_metrics["ROC-AUC Mean"]
    - unweighted_metrics["ROC-AUC Mean"]
)

pr_auc_difference = (
    weighted_metrics["PR-AUC Mean"]
    - unweighted_metrics["PR-AUC Mean"]
)

brier_difference = (
    weighted_metrics["Brier Mean"]
    - unweighted_metrics["Brier Mean"]
)

print(f"ROC-AUC difference: {roc_auc_difference:+.6f}")
print(f"PR-AUC difference:  {pr_auc_difference:+.6f}")
print(f"Brier difference:   {brier_difference:+.6f}")

## 8. Probability calibration experiments

Sigmoid and isotonic calibration are tried on top of the class-weighted baseline and compared against the raw probabilities.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

print("Calibration tools imported successfully.")

In [ ]:
sigmoid_calibrated_pipeline = CalibratedClassifierCV(
    estimator=baseline_pipeline,
    method="sigmoid",
    cv=5
)

print("Sigmoid-calibrated model created.")

In [ ]:
isotonic_calibrated_pipeline = CalibratedClassifierCV(
    estimator=baseline_pipeline,
    method="isotonic",
    cv=5
)

print("Isotonic-calibrated model created.")

In [ ]:
calibration_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Outer calibration CV created.")

In [ ]:
sigmoid_cv_results = cross_validate(
    estimator=sigmoid_calibrated_pipeline,
    X=X_train,
    y=y_train,
    cv=calibration_cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("Sigmoid calibration evaluation completed.")

In [ ]:
isotonic_cv_results = cross_validate(
    estimator=isotonic_calibrated_pipeline,
    X=X_train,
    y=y_train,
    cv=calibration_cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

print("Isotonic calibration evaluation completed.")

In [ ]:
calibration_comparison_detailed = pd.DataFrame([
    {
        "Model": "Class-weighted Logistic (raw)",
        "ROC-AUC Mean": cv_results["test_roc_auc"].mean(),
        "ROC-AUC Std": cv_results["test_roc_auc"].std(),
        "PR-AUC Mean": cv_results["test_pr_auc"].mean(),
        "PR-AUC Std": cv_results["test_pr_auc"].std(),
        "Brier Mean": -cv_results["test_neg_brier"].mean(),
        "Brier Std": (-cv_results["test_neg_brier"]).std()
    },
    {
        "Model": "Class-weighted Logistic + Sigmoid",
        "ROC-AUC Mean": sigmoid_cv_results["test_roc_auc"].mean(),
        "ROC-AUC Std": sigmoid_cv_results["test_roc_auc"].std(),
        "PR-AUC Mean": sigmoid_cv_results["test_pr_auc"].mean(),
        "PR-AUC Std": sigmoid_cv_results["test_pr_auc"].std(),
        "Brier Mean": -sigmoid_cv_results["test_neg_brier"].mean(),
        "Brier Std": (-sigmoid_cv_results["test_neg_brier"]).std()
    },
    {
        "Model": "Class-weighted Logistic + Isotonic",
        "ROC-AUC Mean": isotonic_cv_results["test_roc_auc"].mean(),
        "ROC-AUC Std": isotonic_cv_results["test_roc_auc"].std(),
        "PR-AUC Mean": isotonic_cv_results["test_pr_auc"].mean(),
        "PR-AUC Std": isotonic_cv_results["test_pr_auc"].std(),
        "Brier Mean": -isotonic_cv_results["test_neg_brier"].mean(),
        "Brier Std": (-isotonic_cv_results["test_neg_brier"]).std()
    }
])

calibration_comparison_detailed

## 9. Credit-scorecard-style model: a leakage-safe Weight-of-Evidence (WoE) transformer

A custom `WoETransformer` is implemented from scratch below. It's fit only inside each training fold (never on validation/test data), assigns dedicated bins to missing values and to the special delinquency codes, and reports Information Value (IV) per feature — the standard way credit scorecards justify which variables they keep.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

print("WoE transformer dependencies imported successfully.")

In [ ]:
SPECIAL_VALUE_MAP = {
    "NumberOfTime30-59DaysPastDueNotWorse": [96, 98],
    "NumberOfTimes90DaysLate": [96, 98],
    "NumberOfTime60-89DaysPastDueNotWorse": [96, 98]
}


class WoETransformer(BaseEstimator, TransformerMixin):
    """
    Leakage-safe Weight of Evidence (WoE) transformer.

    Target convention:
        0 = Good
        1 = Bad / Default

    WoE:
        ln(%Good / %Bad)

    Special values and missing values receive dedicated bins.
    """

    def __init__(
        self,
        n_bins=10,
        smoothing=0.5,
        special_values=None,
        max_exact_unique=20
    ):
        self.n_bins = n_bins
        self.smoothing = smoothing
        self.special_values = special_values or {}
        self.max_exact_unique = max_exact_unique

    def _validate_X(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        return X.copy()

    def _make_bin_labels(self, X, feature):
        """
        Convert raw feature values into learned bin labels.
        """
        values = X[feature]
        result = pd.Series(index=values.index, dtype="object")

        # --------------------------------------------------
        # 1. Missing values
        # --------------------------------------------------
        missing_mask = values.isna()
        result.loc[missing_mask] = "__MISSING__"

        # --------------------------------------------------
        # 2. Special values
        # --------------------------------------------------
        special_values = self.special_values.get(feature, [])

        for special_value in special_values:
            mask = values.eq(special_value)
            result.loc[mask] = f"__SPECIAL_{special_value}__"

        # --------------------------------------------------
        # 3. Legitimate values
        # --------------------------------------------------
        normal_mask = ~missing_mask

        if special_values:
            normal_mask &= ~values.isin(special_values)

        normal_values = values.loc[normal_mask]

        if normal_values.empty:
            return result

        # --------------------------------------------------
        # Exact-value binning for low-cardinality variables
        # --------------------------------------------------
        if feature in self.binning_type_ and self.binning_type_[feature] == "exact":

            known_values = self.known_values_[feature]

            for idx, value in normal_values.items():

                if value in self.known_value_to_label_[feature]:
                    result.loc[idx] = self.known_value_to_label_[feature][value]

                else:
                    # Unseen value in validation/test:
                    # map to nearest known legitimate value.
                    nearest_value = known_values[
                        np.argmin(np.abs(known_values - value))
                    ]

                    result.loc[idx] = (
                        self.known_value_to_label_[feature][nearest_value]
                    )

            return result

        # --------------------------------------------------
        # Quantile binning for higher-cardinality variables
        # --------------------------------------------------
        edges = self.bin_edges_[feature]

        result.loc[normal_mask] = pd.cut(
            normal_values,
            bins=edges,
            include_lowest=True,
            duplicates="drop"
        ).astype(str)

        return result

    def fit(self, X, y):
        X = self._validate_X(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)

        if len(X) != len(y):
            raise ValueError("X and y must have the same number of rows.")

        self.feature_names_in_ = list(X.columns)

        self.binning_type_ = {}
        self.known_values_ = {}
        self.known_value_to_label_ = {}
        self.bin_edges_ = {}
        self.woe_maps_ = {}
        self.iv_ = {}

        for feature in self.feature_names_in_:

            values = X[feature]

            # --------------------------------------------------
            # Identify legitimate values
            # --------------------------------------------------
            special_values = self.special_values.get(feature, [])

            normal_mask = ~values.isna()

            if special_values:
                normal_mask &= ~values.isin(special_values)

            normal_values = values.loc[normal_mask]

            unique_values = np.sort(
                normal_values.dropna().unique()
            )

            # --------------------------------------------------
            # Choose binning strategy
            # --------------------------------------------------
            if len(unique_values) <= self.max_exact_unique:

                self.binning_type_[feature] = "exact"

                self.known_values_[feature] = (
                    np.asarray(unique_values, dtype=float)
                )

                self.known_value_to_label_[feature] = {
                    value: f"VALUE_{value}"
                    for value in unique_values
                }

                # No numeric bin edges needed.
                self.bin_edges_[feature] = None

            else:

                self.binning_type_[feature] = "quantile"

                # Quantile boundaries.
                quantiles = np.linspace(
                    0,
                    1,
                    self.n_bins + 1
                )

                edges = np.quantile(
                    normal_values.dropna(),
                    quantiles
                )

                edges = np.unique(edges)

                # If quantile binning collapses because of extreme
                # concentration, fall back to equal-width bins.
                if len(edges) < 3:

                    min_value = normal_values.min()
                    max_value = normal_values.max()

                    edges = np.linspace(
                        min_value,
                        max_value,
                        self.n_bins + 1
                    )

                    edges = np.unique(edges)

                # Infinite outer boundaries make future values
                # outside the training range safe.
                edges[0] = -np.inf
                edges[-1] = np.inf

                self.bin_edges_[feature] = edges

            # --------------------------------------------------
            # Create training bin labels
            # --------------------------------------------------
            temp_X = X[[feature]].copy()

            bin_labels = self._make_bin_labels(
                temp_X,
                feature
            )

            temp = pd.DataFrame({
                "bin": bin_labels,
                "target": y
            })

            # --------------------------------------------------
            # Calculate Good / Bad counts
            # --------------------------------------------------
            all_bins = sorted(
                temp["bin"].dropna().unique().tolist()
            )

            good_counts = (
                temp.loc[temp["target"] == 0]
                .groupby("bin")
                .size()
                .reindex(all_bins, fill_value=0)
            )

            bad_counts = (
                temp.loc[temp["target"] == 1]
                .groupby("bin")
                .size()
                .reindex(all_bins, fill_value=0)
            )

            # --------------------------------------------------
            # Apply smoothing
            # --------------------------------------------------
            n_bins_observed = len(all_bins)

            total_good = (
                good_counts.sum()
                + self.smoothing * n_bins_observed
            )

            total_bad = (
                bad_counts.sum()
                + self.smoothing * n_bins_observed
            )

            good_dist = (
                good_counts + self.smoothing
            ) / total_good

            bad_dist = (
                bad_counts + self.smoothing
            ) / total_bad

            # --------------------------------------------------
            # WoE
            # --------------------------------------------------
            woe = np.log(
                good_dist / bad_dist
            )

            # --------------------------------------------------
            # Information Value
            # --------------------------------------------------
            iv = (
                (good_dist - bad_dist) * woe
            ).sum()

            self.woe_maps_[feature] = woe.to_dict()
            self.iv_[feature] = float(iv)

        return self

    def transform(self, X):
        X = self._validate_X(X)

        transformed = pd.DataFrame(
            index=X.index
        )

        for feature in self.feature_names_in_:

            if feature not in X.columns:
                raise ValueError(
                    f"Missing required feature: {feature}"
                )

            bin_labels = self._make_bin_labels(
                X[[feature]],
                feature
            )

            transformed[feature] = (
                bin_labels
                .map(self.woe_maps_[feature])
                .fillna(0.0)
            )

        return transformed

    def get_feature_names_out(self, input_features=None):
        return np.asarray(
            self.feature_names_in_,
            dtype=object
        )

In [ ]:
woe_transformer = WoETransformer(
    n_bins=10,
    smoothing=0.5,
    special_values=SPECIAL_VALUE_MAP,
    max_exact_unique=20
)

woe_transformer.fit(
    X_train,
    y_train
)

X_train_woe = woe_transformer.transform(X_train)

In [ ]:
print("WoE-transformed shape:", X_train_woe.shape)
print("NaN values:", X_train_woe.isna().sum().sum())
print("Infinite values:", np.isinf(X_train_woe.to_numpy()).sum())

print("\nInformation Value:")
iv_table = (
    pd.Series(woe_transformer.iv_)
    .sort_values(ascending=False)
)

display(iv_table)

In [ ]:
for feature in SPECIAL_VALUE_MAP:

    print("\n" + "=" * 70)
    print(feature)
    print("=" * 70)

    print("Binning strategy:",
          woe_transformer.binning_type_[feature])

    print(
        "Number of learned legitimate values:",
        len(woe_transformer.known_values_[feature])
    )

    print(
        "WoE bins:",
        len(woe_transformer.woe_maps_[feature])
    )

    print(
        "WoE map:",
        woe_transformer.woe_maps_[feature]
    )

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


scorecard_classifier = LogisticRegression(
    class_weight=None,
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

scorecard_pipeline = Pipeline(
    steps=[
        (
            "woe",
            WoETransformer(
                n_bins=10,
                smoothing=0.5,
                special_values=SPECIAL_VALUE_MAP,
                max_exact_unique=20
            )
        ),
        (
            "classifier",
            scorecard_classifier
        )
    ]
)

In [ ]:
fitted_scorecard = scorecard_pipeline.fit(X_train, y_train)

classifier = fitted_scorecard.named_steps["classifier"]

coef_table = pd.DataFrame({
    "Feature": MODEL_COLS,
    "Coefficient": classifier.coef_[0]
})

coef_table["Abs_Coefficient"] = coef_table["Coefficient"].abs()

coef_table = (
    coef_table
    .sort_values("Abs_Coefficient", ascending=False)
    .drop(columns="Abs_Coefficient")
)

display(coef_table)

In [ ]:
iv_table = (
    pd.Series(
        fitted_scorecard.named_steps["woe"].iv_
    )
    .sort_values(ascending=False)
)

display(iv_table)

In [ ]:
for feature in [
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTimes90DaysLate",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse"
]:

    print("\n" + "=" * 80)
    print(feature)
    print("=" * 80)

    woe_map = fitted_scorecard.named_steps["woe"].woe_maps_[feature]

    for bin_name, woe_value in woe_map.items():
        print(f"{str(bin_name):35s} WoE = {woe_value: .4f}")

### 9.1 Coarse-binning the delinquency counters

Reduces bin sparsity for the three delinquency-count features before WoE encoding.

In [ ]:
DELINQUENCY_FEATURES = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

for feature in DELINQUENCY_FEATURES:

    print("\n" + "=" * 80)
    print(feature)
    print("=" * 80)

    temp = pd.DataFrame({
        "value": X_train[feature],
        "target": y_train
    })

    temp["group"] = temp["value"].apply(
        lambda x:
            "96" if x == 96 else
            "98" if x == 98 else
            "3+" if x >= 3 else
            str(int(x))
    )

    summary = (
        temp.groupby("group")["target"]
        .agg(
            applicants="count",
            defaults="sum",
            default_rate="mean"
        )
        .reindex(["0", "1", "2", "3+", "96", "98"])
    )

    display(summary)

In [ ]:
COARSE_DELINQUENCY_FEATURES = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]


def apply_coarse_delinquency_bins(X):
    """
    Coarse-bin the three delinquency count features.

    Legitimate values:
        0 -> 0
        1 -> 1
        2 -> 2
        >=3 -> 3

    Special values:
        96 -> retained
        98 -> retained

    Missing values:
        retained as NaN
    """

    X = X.copy()

    for feature in COARSE_DELINQUENCY_FEATURES:

        normal_mask = (
            X[feature].notna()
            & ~X[feature].isin([96, 98])
        )

        X.loc[normal_mask, feature] = np.minimum(
            X.loc[normal_mask, feature],
            3
        )

    return X

In [ ]:
X_train_coarse = apply_coarse_delinquency_bins(X_train)

for feature in COARSE_DELINQUENCY_FEATURES:

    print("\n" + "=" * 70)
    print(feature)
    print("=" * 70)

    print(
        X_train_coarse[feature]
        .value_counts(dropna=False)
        .sort_index()
    )

In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
scorecard_pipeline = Pipeline(
    steps=[
        (
            "coarse_binning",
            FunctionTransformer(
                apply_coarse_delinquency_bins,
                validate=False
            )
        ),
        (
            "woe",
            WoETransformer(
                n_bins=10,
                smoothing=0.5,
                special_values=SPECIAL_VALUE_MAP,
                max_exact_unique=20
            )
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight=None,
                max_iter=2000,
                solver="lbfgs",
                random_state=42
            )
        )
    ]
)

In [ ]:
scorecard_coarse_cv_results = cross_validate(
    scorecard_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
scorecard_coarse_summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score"
    ],
    "Mean": [
        scorecard_coarse_cv_results["test_roc_auc"].mean(),
        scorecard_coarse_cv_results["test_pr_auc"].mean(),
        -scorecard_coarse_cv_results["test_neg_brier"].mean()
    ],
    "Std": [
        scorecard_coarse_cv_results["test_roc_auc"].std(),
        scorecard_coarse_cv_results["test_pr_auc"].std(),
        scorecard_coarse_cv_results["test_neg_brier"].std()
    ]
})

display(scorecard_coarse_summary)

## 10. Gradient-boosted model: HistGradientBoostingClassifier

Same special-code handling as the scorecard (indicator features for codes 96/98, then treated as missing), but no WoE binning — the tree model can use the raw values directly.

In [ ]:
TREE_SPECIAL_FEATURES = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]


def prepare_tree_features(X):
    X = X.copy()

    for feature in TREE_SPECIAL_FEATURES:

        # Explicit indicators for special codes
        X[f"{feature}_special_96"] = (
            X[feature] == 96
        ).astype(int)

        X[f"{feature}_special_98"] = (
            X[feature] == 98
        ).astype(int)

        # Treat special codes as missing rather than
        # literal delinquency counts
        X.loc[
            X[feature].isin([96, 98]),
            feature
        ] = np.nan

    return X

In [ ]:
X_train_tree = prepare_tree_features(X_train)

print("Original shape:", X_train.shape)
print("Tree feature shape:", X_train_tree.shape)

print("\nSpecial-code columns:")
display(
    X_train_tree[
        [
            "NumberOfTime30-59DaysPastDueNotWorse_special_96",
            "NumberOfTime30-59DaysPastDueNotWorse_special_98",
            "NumberOfTimes90DaysLate_special_96",
            "NumberOfTimes90DaysLate_special_98",
            "NumberOfTime60-89DaysPastDueNotWorse_special_96",
            "NumberOfTime60-89DaysPastDueNotWorse_special_98"
        ]
    ].sum()
)

print("\nRemaining 96/98 values:")
for feature in TREE_SPECIAL_FEATURES:
    print(
        feature,
        "96/98 remaining =",
        X_train_tree[feature].isin([96, 98]).sum()
    )

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier


hgb_baseline = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42
)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight


hgb_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("Sample weights:")
print(
    pd.Series(hgb_sample_weights)
    .describe()
)

print("\nMean weight for non-defaults:",
      hgb_sample_weights[y_train.to_numpy() == 0].mean())

print("Mean weight for defaults:",
      hgb_sample_weights[y_train.to_numpy() == 1].mean())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


hgb_pipeline = Pipeline(
    steps=[
        (
            "tree_preprocessing",
            FunctionTransformer(
                prepare_tree_features,
                validate=False
            )
        ),
        (
            "classifier",
            hgb_baseline
        )
    ]
)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)


def evaluate_hgb_cv(
    X,
    y,
    cv,
    model
):

    results = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_fold_train = X.iloc[train_idx]
        X_fold_valid = X.iloc[valid_idx]

        y_fold_train = y.iloc[train_idx]
        y_fold_valid = y.iloc[valid_idx]

        # Compute class-balanced weights ONLY from
        # the current training fold.
        fold_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_fold_train
        )

        fold_model = clone(model)

        fold_model.fit(
            X_fold_train,
            y_fold_train,
            classifier__sample_weight=fold_weights
        )

        probabilities = fold_model.predict_proba(
            X_fold_valid
        )[:, 1]

        results.append({
            "fold": fold,
            "roc_auc": roc_auc_score(
                y_fold_valid,
                probabilities
            ),
            "pr_auc": average_precision_score(
                y_fold_valid,
                probabilities
            ),
            "brier": brier_score_loss(
                y_fold_valid,
                probabilities
            )
        })

    return pd.DataFrame(results)

In [ ]:
hgb_cv_results = evaluate_hgb_cv(
    X_train,
    y_train,
    cv,
    hgb_pipeline
)

display(hgb_cv_results)

print("\nMean metrics:")
print(
    hgb_cv_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean()
)

print("\nStandard deviation:")
print(
    hgb_cv_results[
        ["roc_auc", "pr_auc", "brier"]
    ].std()
)

In [ ]:
def evaluate_hgb_cv_unweighted(
    X,
    y,
    cv,
    model
):

    results = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_fold_train = X.iloc[train_idx]
        X_fold_valid = X.iloc[valid_idx]

        y_fold_train = y.iloc[train_idx]
        y_fold_valid = y.iloc[valid_idx]

        fold_model = clone(model)

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        probabilities = fold_model.predict_proba(
            X_fold_valid
        )[:, 1]

        results.append({
            "fold": fold,
            "roc_auc": roc_auc_score(
                y_fold_valid,
                probabilities
            ),
            "pr_auc": average_precision_score(
                y_fold_valid,
                probabilities
            ),
            "brier": brier_score_loss(
                y_fold_valid,
                probabilities
            )
        })

    return pd.DataFrame(results)

In [ ]:
hgb_unweighted_results = evaluate_hgb_cv_unweighted(
    X_train,
    y_train,
    cv,
    hgb_pipeline
)

display(hgb_unweighted_results)

print("\nMean metrics:")
print(
    hgb_unweighted_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean()
)

print("\nStandard deviation:")
print(
    hgb_unweighted_results[
        ["roc_auc", "pr_auc", "brier"]
    ].std()
)

In [ ]:
hgb_unweighted = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42
)

hgb_pipeline = Pipeline(
    steps=[
        (
            "tree_preprocessing",
            FunctionTransformer(
                prepare_tree_features,
                validate=False
            )
        ),
        (
            "classifier",
            hgb_unweighted
        )
    ]
)

## 11. Hyperparameter tuning with Optuna

20 trials, each scored by the same fold-respecting CV loop used above (`evaluate_hgb_cv_unweighted`), optimizing mean CV ROC-AUC.

In [ ]:
try:
    import optuna
    print("Optuna version:", optuna.__version__)
except ImportError:
    print("Optuna is not installed.")

In [ ]:
%pip install optuna

In [ ]:
import optuna

print("Optuna version:", optuna.__version__)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import roc_auc_score


def objective(trial):

    model = Pipeline(
        steps=[
            (
                "tree_preprocessing",
                FunctionTransformer(
                    prepare_tree_features,
                    validate=False
                )
            ),
            (
                "classifier",
                HistGradientBoostingClassifier(
                    learning_rate=trial.suggest_float(
                        "learning_rate",
                        0.01,
                        0.2,
                        log=True
                    ),
                    max_iter=trial.suggest_int(
                        "max_iter",
                        150,
                        500
                    ),
                    max_leaf_nodes=trial.suggest_int(
                        "max_leaf_nodes",
                        15,
                        63
                    ),
                    min_samples_leaf=trial.suggest_int(
                        "min_samples_leaf",
                        20,
                        150
                    ),
                    l2_regularization=trial.suggest_float(
                        "l2_regularization",
                        1e-3,
                        10.0,
                        log=True
                    ),
                    random_state=42
                )
            )
        ]
    )

    fold_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        fold_model = clone(model)

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        probabilities = fold_model.predict_proba(
            X_fold_valid
        )[:, 1]

        fold_scores.append(
            roc_auc_score(
                y_fold_valid,
                probabilities
            )
        )

    return np.mean(fold_scores)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="credit_risk_hgb"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

In [ ]:
print("Best ROC-AUC:", study.best_value)

print("\nBest parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

In [ ]:
best_hgb_pipeline = Pipeline(
    steps=[
        (
            "tree_preprocessing",
            FunctionTransformer(
                prepare_tree_features,
                validate=False
            )
        ),
        (
            "classifier",
            HistGradientBoostingClassifier(
                **study.best_params,
                random_state=42
            )
        )
    ]
)

In [ ]:
tuned_hgb_results = evaluate_hgb_cv_unweighted(
    X_train,
    y_train,
    cv,
    best_hgb_pipeline
)

display(tuned_hgb_results)

print("\nMean metrics:")
print(
    tuned_hgb_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean()
)

print("\nStandard deviation:")
print(
    tuned_hgb_results[
        ["roc_auc", "pr_auc", "brier"]
    ].std()
)

## 12. Final model

The tuned pipeline (best Optuna params) fit on the full training set. This is the model object evaluated against the held-out test set below, and the one ultimately saved to `model_artifacts/`.

In [ ]:
final_hgb_pipeline = Pipeline(
    steps=[
        (
            "tree_preprocessing",
            FunctionTransformer(
                prepare_tree_features,
                validate=False
            )
        ),
        (
            "classifier",
            HistGradientBoostingClassifier(
                **study.best_params,
                random_state=42
            )
        )
    ]
)

final_hgb_pipeline.fit(
    X_train,
    y_train
)

## 13. Held-out test set evaluation

The test set was untouched by every step above — no preprocessing fit, no model selection, no hyperparameter tuning ever saw it. These are the numbers reported in the README.

In [ ]:
test_probabilities = final_hgb_pipeline.predict_proba(
    X_test
)[:, 1]

print("Test predictions generated:", len(test_probabilities))
print("Minimum probability:", test_probabilities.min())
print("Maximum probability:", test_probabilities.max())

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

test_brier = brier_score_loss(
    y_test,
    test_probabilities
)

print("FINAL TEST-SET RESULTS")
print("=" * 40)
print(f"ROC-AUC : {test_roc_auc:.6f}")
print(f"PR-AUC  : {test_pr_auc:.6f}")
print(f"Brier   : {test_brier:.6f}")

In [ ]:
from sklearn.metrics import roc_auc_score


def calculate_ks(y_true, probabilities):
    """
    Calculate the Kolmogorov-Smirnov statistic
    for binary credit-risk predictions.

    Higher KS indicates greater separation between
    the bad and good distributions.
    """

    evaluation = pd.DataFrame({
        "y": np.asarray(y_true),
        "probability": np.asarray(probabilities)
    })

    evaluation = evaluation.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    total_bad = (evaluation["y"] == 1).sum()
    total_good = (evaluation["y"] == 0).sum()

    evaluation["cum_bad"] = (
        (evaluation["y"] == 1).cumsum()
        / total_bad
    )

    evaluation["cum_good"] = (
        (evaluation["y"] == 0).cumsum()
        / total_good
    )

    evaluation["ks"] = (
        evaluation["cum_bad"]
        - evaluation["cum_good"]
    ).abs()

    ks_statistic = evaluation["ks"].max()

    return ks_statistic


test_gini = (
    2 * test_roc_auc - 1
)

test_ks = calculate_ks(
    y_test,
    test_probabilities
)

print("FINAL TEST-SET CREDIT METRICS")
print("=" * 45)
print(f"ROC-AUC : {test_roc_auc:.6f}")
print(f"Gini    : {test_gini:.6f}")
print(f"KS      : {test_ks:.6f}")
print(f"PR-AUC  : {test_pr_auc:.6f}")
print(f"Brier   : {test_brier:.6f}")

### 13.1 Bootstrap 95% confidence intervals

The model is not retrained here — only the fixed test-set predictions are resampled, 2,000 times, to get an uncertainty range around each point estimate.

In [ ]:
from sklearn.utils import resample


def bootstrap_metric_ci(
    y_true,
    probabilities,
    metric_fn,
    n_bootstraps=2000,
    random_state=42
):
    """
    Bootstrap a metric on fixed test-set predictions.

    The model is NOT retrained.
    Only the test observations are resampled.
    """

    rng = np.random.RandomState(random_state)

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    scores = []

    n = len(y_true)

    for _ in range(n_bootstraps):

        indices = rng.randint(
            0,
            n,
            size=n
        )

        y_sample = y_true[indices]
        probability_sample = probabilities[indices]

        # Skip degenerate bootstrap samples
        # where only one target class appears.
        if len(np.unique(y_sample)) < 2:
            continue

        scores.append(
            metric_fn(
                y_sample,
                probability_sample
            )
        )

    scores = np.asarray(scores)

    lower = np.percentile(
        scores,
        2.5
    )

    upper = np.percentile(
        scores,
        97.5
    )

    return {
        "estimate": metric_fn(
            y_true,
            probabilities
        ),
        "lower_95": lower,
        "upper_95": upper,
        "n_bootstraps": len(scores)
    }

In [ ]:
roc_auc_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    roc_auc_score,
    n_bootstraps=2000,
    random_state=42
)

pr_auc_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    average_precision_score,
    n_bootstraps=2000,
    random_state=42
)

brier_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    brier_score_loss,
    n_bootstraps=2000,
    random_state=42
)

bootstrap_summary = pd.DataFrame([
    {
        "Metric": "ROC-AUC",
        "Estimate": roc_auc_ci["estimate"],
        "CI Lower": roc_auc_ci["lower_95"],
        "CI Upper": roc_auc_ci["upper_95"]
    },
    {
        "Metric": "PR-AUC",
        "Estimate": pr_auc_ci["estimate"],
        "CI Lower": pr_auc_ci["lower_95"],
        "CI Upper": pr_auc_ci["upper_95"]
    },
    {
        "Metric": "Brier",
        "Estimate": brier_ci["estimate"],
        "CI Lower": brier_ci["lower_95"],
        "CI Upper": brier_ci["upper_95"]
    }
])

display(bootstrap_summary)

### 13.2 Calibration curve

Checks whether a predicted probability of X% actually corresponds to an observed default rate of about X% on the test set.

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt


fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(7, 6))

plt.plot(
    mean_predicted_value,
    fraction_of_positives,
    marker="o",
    label="HGB"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default rate")
plt.title("Calibration Curve — Final HGB Model")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 14. Out-of-fold predictions on the training set

Used for threshold selection below, so the test set is never used to choose a decision policy.

In [ ]:
from sklearn.base import clone


def generate_oof_predictions(
    X,
    y,
    cv,
    model
):
    """
    Generate leakage-safe out-of-fold probabilities.

    Every observation receives a prediction from a model
    that was NOT trained on that observation.
    """

    oof_probabilities = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_fold_train = X.iloc[train_idx]
        X_fold_valid = X.iloc[valid_idx]

        y_fold_train = y.iloc[train_idx]

        fold_model = clone(model)

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        oof_probabilities[valid_idx] = (
            fold_model.predict_proba(
                X_fold_valid
            )[:, 1]
        )

        print(
            f"Fold {fold} complete"
        )

    return oof_probabilities

In [ ]:
oof_probabilities = generate_oof_predictions(
    X_train,
    y_train,
    cv,
    best_hgb_pipeline
)

print(
    "OOF predictions:",
    len(oof_probabilities)
)

print(
    "Minimum:",
    oof_probabilities.min()
)

print(
    "Maximum:",
    oof_probabilities.max()
)

### 14.1 Threshold sweep at a fixed cost ratio

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score
)


thresholds = np.arange(
    0.05,
    0.56,
    0.05
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train,
        predictions
    ).ravel()

    threshold_results.append({
        "threshold": threshold,
        "approval_rate": (predictions == 0).mean(),
        "predicted_default_rate": predictions.mean(),
        "precision": precision_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "false_positive_rate": fp / (fp + tn),
        "false_negative_rate": fn / (fn + tp)
    })

threshold_table = pd.DataFrame(
    threshold_results
)

display(threshold_table)

### 14.2 Threshold sensitivity across different cost-ratio assumptions

Shows how the optimal threshold moves as the assumed false-negative/false-positive cost ratio changes — making explicit that the threshold is a policy choice, not a fixed property of the model.

In [ ]:
cost_ratios = [1, 2, 5, 10, 20]

cost_results = []

for fn_cost_ratio in cost_ratios:

    for threshold in np.arange(0.01, 1.00, 0.01):

        predictions = (
            oof_probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_train,
            predictions
        ).ravel()

        total_cost = (
            fp * 1
            + fn * fn_cost_ratio
        )

        cost_per_applicant = (
            total_cost / len(y_train)
        )

        cost_results.append({
            "FN_to_FP_cost_ratio": fn_cost_ratio,
            "threshold": threshold,
            "cost_per_applicant": cost_per_applicant,
            "approval_rate": (predictions == 0).mean(),
            "recall": recall_score(
                y_train,
                predictions,
                zero_division=0
            )
        })

cost_df = pd.DataFrame(cost_results)

optimal_by_ratio = (
    cost_df
    .loc[
        cost_df.groupby(
            "FN_to_FP_cost_ratio"
        )["cost_per_applicant"]
        .idxmin()
    ]
    .sort_values("FN_to_FP_cost_ratio")
)

display(optimal_by_ratio)

## 15. Explainability: SHAP

`TreeExplainer` computes exact Shapley values for the HistGradientBoosting model. This is also what `src/credit_risk/predictor.py` uses at inference time to generate the `reasons` field in the API response.

In [ ]:
try:
    import shap
    print("SHAP version:", shap.__version__)
except ImportError:
    print("SHAP is not installed.")

In [ ]:
# Extract the fitted preprocessing and classifier
tree_preprocessor = final_hgb_pipeline.named_steps[
    "tree_preprocessing"
]

final_hgb_model = final_hgb_pipeline.named_steps[
    "classifier"
]

# Transform training and test data exactly as the final pipeline does
X_train_tree = tree_preprocessor.transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)

print("Transformed training shape:", X_train_tree.shape)
print("Transformed test shape:", X_test_tree.shape)

In [ ]:
shap_feature_names = list(X_train_tree.columns)

print("SHAP features:")
for i, feature in enumerate(shap_feature_names):
    print(f"{i}: {feature}")

In [ ]:
import shap

explainer = shap.TreeExplainer(
    final_hgb_model
)

print("SHAP explainer created.")

In [ ]:
SHAP_SAMPLE_SIZE = 3000

X_shap = X_test_tree.iloc[
    :SHAP_SAMPLE_SIZE
].copy()

shap_values = explainer(
    X_shap
)

print("SHAP values shape:", shap_values.values.shape)

In [ ]:
shap_importance = pd.DataFrame({
    "Feature": shap_feature_names,
    "Mean_Absolute_SHAP": np.abs(
        shap_values.values
    ).mean(axis=0)
})

shap_importance = (
    shap_importance
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

display(shap_importance)

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values.values,
    X_shap,
    feature_names=shap_feature_names,
    show=False
)

plt.title(
    "Global SHAP Feature Importance — Final HGB Model"
)

plt.tight_layout()
plt.show()

### 15.1 Per-applicant reason codes

Walks through generating a human-readable explanation for one specific applicant (the highest-risk applicant in the test set), building up to the same `generate_reason_codes` logic the API uses.

In [ ]:
highest_risk_position = np.argmax(
    test_probabilities
)

print(
    "Test-set position:",
    highest_risk_position
)

print(
    "Predicted default probability:",
    test_probabilities[
        highest_risk_position
    ]
)

print(
    "Actual target:",
    y_test.iloc[
        highest_risk_position
    ]
)

In [ ]:
# Get the selected high-risk applicant
local_features = X_test_tree.iloc[[highest_risk_position]]

# Calculate SHAP values specifically for this applicant
local_shap = explainer(local_features)

local_shap_values = local_shap.values[0]

print("Applicant position:", highest_risk_position)
print(
    "Predicted default probability:",
    test_probabilities[highest_risk_position]
)
print(
    "Actual target:",
    y_test.iloc[highest_risk_position]
)

In [ ]:
local_explanation = pd.DataFrame({
    "Feature": shap_feature_names,
    "Value": local_features.iloc[0].values,
    "SHAP_Contribution": local_shap_values
})

local_explanation["Absolute_SHAP"] = (
    local_explanation["SHAP_Contribution"].abs()
)

local_explanation["Direction"] = np.where(
    local_explanation["SHAP_Contribution"] > 0,
    "Higher default risk",
    "Lower default risk"
)

local_explanation = (
    local_explanation
    .sort_values(
        "Absolute_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

display(local_explanation)

In [ ]:
reason_codes = (
    local_explanation
    .loc[
        local_explanation["Absolute_SHAP"] > 0,
        [
            "Feature",
            "Value",
            "SHAP_Contribution",
            "Direction"
        ]
    ]
    .head(5)
)

display(reason_codes)

In [ ]:
REASON_CODE_MAP = {
    "RevolvingUtilizationOfUnsecuredLines":
        "High revolving credit utilization",

    "NumberOfTime30-59DaysPastDueNotWorse":
        "Multiple 30–59 day delinquencies",

    "NumberOfTimes90DaysLate":
        "Frequent 90+ day delinquencies",

    "NumberOfTime60-89DaysPastDueNotWorse":
        "Multiple 60–89 day delinquencies",

    "DebtRatio":
        "High debt ratio",

    "MonthlyIncome":
        "Income information is missing",

    "NumberOfOpenCreditLinesAndLoans":
        "Credit-line profile contributes to risk",

    "NumberRealEstateLoansOrLines":
        "Real-estate loan profile contributes to risk",

    "NumberOfDependents":
        "Number of dependents contributes to risk"
}

In [ ]:
positive_reasons = (
    local_explanation[
        local_explanation["SHAP_Contribution"] > 0
    ]
    .copy()
)

positive_reasons["Reason_Code"] = (
    positive_reasons["Feature"]
    .map(REASON_CODE_MAP)
)

positive_reasons = (
    positive_reasons[
        positive_reasons["Reason_Code"].notna()
    ]
    .head(5)
)

display(
    positive_reasons[
        [
            "Feature",
            "Value",
            "SHAP_Contribution",
            "Reason_Code"
        ]
    ]
)

In [ ]:
def generate_reason_codes(
    model,
    X_applicant,
    feature_names,
    reason_code_map,
    top_n=5
):
    """
    Generate human-readable SHAP reason codes
    for one applicant.

    Positive SHAP  -> higher default risk
    Negative SHAP  -> lower default risk
    Zero SHAP      -> no material contribution
    """

    # Create SHAP explainer
    explainer = shap.TreeExplainer(
        model
    )

    # Calculate SHAP values
    shap_result = explainer(
        X_applicant
    )

    shap_values_local = shap_result.values[0]

    # Build explanation table
    explanation = pd.DataFrame({
        "Feature": feature_names,
        "Value": X_applicant.iloc[0].values,
        "SHAP_Contribution": shap_values_local
    })

    explanation["Absolute_SHAP"] = (
        explanation["SHAP_Contribution"].abs()
    )

    # Correctly classify contribution direction
    explanation["Direction"] = np.select(
        [
            explanation["SHAP_Contribution"] > 0,
            explanation["SHAP_Contribution"] < 0,
            explanation["SHAP_Contribution"] == 0
        ],
        [
            "Higher default risk",
            "Lower default risk",
            "No material contribution"
        ],
        default="No material contribution"
    )

    # Map technical features to human-readable reasons
    explanation["Reason_Code"] = (
        explanation["Feature"]
        .map(reason_code_map)
    )

    # Only keep features with a non-zero
    # contribution and a defined reason code
    positive_reasons = (
        explanation[
            (explanation["SHAP_Contribution"] > 0)
            & (explanation["Reason_Code"].notna())
        ]
        .sort_values(
            "Absolute_SHAP",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return explanation, positive_reasons

In [ ]:
local_explanation, reason_codes = generate_reason_codes(
    model=final_hgb_model,
    X_applicant=local_features,
    feature_names=shap_feature_names,
    reason_code_map=REASON_CODE_MAP,
    top_n=5
)

display(reason_codes)

In [ ]:
applicant_explanation = {
    "default_probability": float(
        test_probabilities[highest_risk_position]
    ),
    "reason_codes": reason_codes[
        "Reason_Code"
    ].tolist()
}

applicant_explanation

## 16. Fairness audit by age group

`age` was never a model feature (Section 4) — this section checks whether outcomes are nonetheless skewed across age groups, using the audit attribute carried through since the train/test split.

In [ ]:
audit_test["age"].describe()

In [ ]:
print(
    audit_test["age"]
    .value_counts()
    .sort_index()
)

In [ ]:
audit_test_fairness = audit_test.copy()

audit_test_fairness["age_group"] = pd.cut(
    audit_test_fairness["age"],
    bins=[-np.inf, 24, 34, 44, 54, 64, np.inf],
    labels=[
        "<25",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
    ]
)

print("Age-group counts:")
display(
    audit_test_fairness["age_group"]
    .value_counts()
    .sort_index()
)

In [ ]:
fairness_df = pd.DataFrame({
    "y_true": y_test.to_numpy(),
    "predicted_probability": test_probabilities,
    "age": audit_test_fairness["age"].to_numpy(),
    "age_group": audit_test_fairness["age_group"].to_numpy()
})

print("Rows:", len(fairness_df))
print("Missing age groups:", fairness_df["age_group"].isna().sum())

display(
    fairness_df.head()
)

In [ ]:
FAIRNESS_THRESHOLD = 0.17

fairness_df["predicted_default"] = (
    fairness_df["predicted_probability"]
    >= FAIRNESS_THRESHOLD
).astype(int)

fairness_df["approved"] = (
    fairness_df["predicted_default"] == 0
).astype(int)

In [ ]:
print(
    "Illustrative threshold:",
    FAIRNESS_THRESHOLD
)

print(
    "Overall approval rate:",
    fairness_df["approved"].mean()
)

print(
    "Overall predicted default rate:",
    fairness_df["predicted_default"].mean()
)

In [ ]:
fairness_results = []

for group, group_df in fairness_df.groupby(
    "age_group",
    observed=True
):

    approvals = group_df["approved"].sum()
    applicants = len(group_df)

    # True positive rate:
    # among applicants who actually defaulted,
    # how many did the model flag?
    actual_defaults = group_df["y_true"].sum()

    true_positives = (
        (
            (group_df["predicted_default"] == 1)
            & (group_df["y_true"] == 1)
        )
        .sum()
    )

    tpr = (
        true_positives / actual_defaults
        if actual_defaults > 0
        else np.nan
    )

    fairness_results.append({
        "age_group": group,
        "applicants": applicants,
        "approval_rate": (
            approvals / applicants
        ),
        "predicted_default_rate": (
            group_df["predicted_default"].mean()
        ),
        "actual_default_rate": (
            group_df["y_true"].mean()
        ),
        "true_positive_rate": tpr
    })

fairness_results = pd.DataFrame(
    fairness_results
)

display(fairness_results)

In [ ]:
reference_approval_rate = (
    fairness_results["approval_rate"].max()
)

fairness_results["approval_rate_ratio"] = (
    fairness_results["approval_rate"]
    / reference_approval_rate
)

display(
    fairness_results[
        [
            "age_group",
            "applicants",
            "approval_rate",
            "approval_rate_ratio"
        ]
    ]
)

In [ ]:
min_approval_ratio = (
    fairness_results["approval_rate_ratio"].min()
)

print(
    f"Minimum approval-rate ratio: "
    f"{min_approval_ratio:.4f}"
)

print(
    f"80% reference threshold: 0.8000"
)

In [ ]:
tpr_min = fairness_results[
    "true_positive_rate"
].min()

tpr_max = fairness_results[
    "true_positive_rate"
].max()

tpr_difference = tpr_max - tpr_min

print(
    f"Minimum TPR: {tpr_min:.4f}"
)

print(
    f"Maximum TPR: {tpr_max:.4f}"
)

print(
    f"TPR difference: {tpr_difference:.4f}"
)

In [ ]:
group_calibration = []

for group, group_df in fairness_df.groupby(
    "age_group",
    observed=True
):

    # Use 5 quantile bins within each group.
    # This avoids bins with extremely small counts.
    try:
        group_bins = pd.qcut(
            group_df["predicted_probability"],
            q=5,
            duplicates="drop"
        )
    except ValueError:
        continue

    calibration = (
        group_df
        .assign(probability_bin=group_bins)
        .groupby(
            "probability_bin",
            observed=True
        )
        .agg(
            applicants=("y_true", "size"),
            mean_predicted_probability=(
                "predicted_probability",
                "mean"
            ),
            observed_default_rate=(
                "y_true",
                "mean"
            )
        )
        .reset_index()
    )

    calibration["age_group"] = group

    group_calibration.append(calibration)

group_calibration = pd.concat(
    group_calibration,
    ignore_index=True
)

display(group_calibration)

In [ ]:
group_calibration["absolute_calibration_error"] = (
    group_calibration[
        "mean_predicted_probability"
    ]
    -
    group_calibration[
        "observed_default_rate"
    ]
).abs()

calibration_summary = (
    group_calibration
    .groupby("age_group", observed=True)
    .agg(
        max_calibration_error=(
            "absolute_calibration_error",
            "max"
        ),
        mean_calibration_error=(
            "absolute_calibration_error",
            "mean"
        )
    )
    .reset_index()
)

display(calibration_summary)

### 16.1 Cross-check with Fairlearn

The manual fairness calculations above are cross-checked against `fairlearn`'s standard `MetricFrame`, `demographic_parity_difference`, and `equal_opportunity_difference` implementations.

In [ ]:
try:
    import fairlearn
    print("Fairlearn version:", fairlearn.__version__)
except ImportError:
    print("Fairlearn is not installed.")

In [ ]:
%pip install fairlearn

In [ ]:
import fairlearn

print("Fairlearn version:", fairlearn.__version__)

In [ ]:
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    demographic_parity_difference,
    equal_opportunity_difference
)

In [ ]:
fairlearn_metrics = MetricFrame(
    metrics={
        "selection_rate": selection_rate,
        "true_positive_rate": true_positive_rate
    },
    y_true=fairness_df["y_true"],
    y_pred=fairness_df["predicted_default"],
    sensitive_features=fairness_df["age_group"]
)

print("Metrics by age group:")
display(fairlearn_metrics.by_group)

In [ ]:
approval_by_age = MetricFrame(
    metrics={
        "approval_rate": lambda y_true, y_pred: 1 - selection_rate(
            y_true,
            y_pred
        )
    },
    y_true=fairness_df["y_true"],
    y_pred=fairness_df["predicted_default"],
    sensitive_features=fairness_df["age_group"]
)

print("Approval rate by age group:")
display(approval_by_age.by_group)

In [ ]:
dp_difference = demographic_parity_difference(
    y_true=fairness_df["y_true"],
    y_pred=fairness_df["predicted_default"],
    sensitive_features=fairness_df["age_group"]
)

print(
    f"Demographic parity difference "
    f"(default prediction): {dp_difference:.6f}"
)

In [ ]:
eo_difference = equal_opportunity_difference(
    y_true=fairness_df["y_true"],
    y_pred=fairness_df["predicted_default"],
    sensitive_features=fairness_df["age_group"]
)

print(
    f"Equal opportunity difference: "
    f"{eo_difference:.6f}"
)

## 17. Save the final model artifact and configuration

The fitted pipeline (preprocessing + model, as one object) and a config file recording the feature list, decision threshold, and every metric computed above.

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("model_artifacts")
ARTIFACT_DIR.mkdir(
    exist_ok=True
)

print(
    "Artifact directory:",
    ARTIFACT_DIR.resolve()
)

In [ ]:
print(
    "Directory exists:",
    ARTIFACT_DIR.exists()
)

print(
    "Directory contents:",
    list(ARTIFACT_DIR.iterdir())
)

In [ ]:
import joblib

MODEL_PATH = (
    ARTIFACT_DIR / "final_hgb_pipeline.joblib"
)

joblib.dump(
    final_hgb_pipeline,
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

print(
    "File size:",
    MODEL_PATH.stat().st_size / (1024 ** 2),
    "MB"
)

In [ ]:
loaded_hgb_pipeline = joblib.load(
    MODEL_PATH
)

loaded_test_probabilities = (
    loaded_hgb_pipeline
    .predict_proba(X_test)[:, 1]
)

probability_difference = np.max(
    np.abs(
        test_probabilities
        - loaded_test_probabilities
    )
)

print(
    "Maximum probability difference:",
    probability_difference
)

print(
    "Predictions identical:",
    np.allclose(
        test_probabilities,
        loaded_test_probabilities
    )
)

In [ ]:
import json

MODEL_CONFIG = {
    "model_type": "HistGradientBoostingClassifier",
    "target": TARGET_COL,
    "model_features": MODEL_COLS,
    "audit_features": AUDIT_COLS,
    "decision_threshold": 0.17,
    "threshold_scenario": {
        "fn_to_fp_cost_ratio": 5,
        "description": (
            "Illustrative threshold selected using "
            "out-of-fold expected-cost analysis."
        )
    },
    "training": {
        "test_size": 0.20,
        "random_state": 42,
        "cv_folds": 5,
        "stratified": True
    },
    "performance": {
        "test_roc_auc": float(test_roc_auc),
        "test_pr_auc": float(test_pr_auc),
        "test_brier": float(test_brier),
        "test_gini": float(test_gini),
        "test_ks": float(test_ks)
    },
    "bootstrap_95_ci": {
        "roc_auc": [
            float(roc_auc_ci["lower_95"]),
            float(roc_auc_ci["upper_95"])
        ],
        "pr_auc": [
            float(pr_auc_ci["lower_95"]),
            float(pr_auc_ci["upper_95"])
        ],
        "brier": [
            float(brier_ci["lower_95"]),
            float(brier_ci["upper_95"])
        ]
    }
}

CONFIG_PATH = (
    ARTIFACT_DIR / "model_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(
        MODEL_CONFIG,
        f,
        indent=4
    )

print("Configuration saved to:")
print(CONFIG_PATH)

In [ ]:
with open(CONFIG_PATH, "r") as f:
    loaded_config = json.load(f)

print(
    json.dumps(
        loaded_config,
        indent=4
    )
)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Recompute directly from the frozen test predictions
canonical_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

canonical_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

canonical_brier = brier_score_loss(
    y_test,
    test_probabilities
)

canonical_gini = (
    2 * canonical_roc_auc - 1
)

canonical_ks = calculate_ks(
    y_test,
    test_probabilities
)

canonical_metrics = {
    "roc_auc": canonical_roc_auc,
    "pr_auc": canonical_pr_auc,
    "brier": canonical_brier,
    "gini": canonical_gini,
    "ks": canonical_ks
}

print("CANONICAL TEST METRICS")
print("=" * 45)

for metric, value in canonical_metrics.items():
    print(f"{metric:10s}: {value:.9f}")

In [ ]:
print("\nPrediction sanity check")
print("=" * 45)
print("Number of predictions:", len(test_probabilities))
print("Min probability:", test_probabilities.min())
print("Max probability:", test_probabilities.max())

In [ ]:
# Recompute bootstrap confidence intervals
roc_auc_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    roc_auc_score,
    n_bootstraps=2000,
    random_state=42
)

pr_auc_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    average_precision_score,
    n_bootstraps=2000,
    random_state=42
)

brier_ci = bootstrap_metric_ci(
    y_test,
    test_probabilities,
    brier_score_loss,
    n_bootstraps=2000,
    random_state=42
)

canonical_bootstrap_summary = pd.DataFrame([
    {
        "Metric": "ROC-AUC",
        "Estimate": roc_auc_ci["estimate"],
        "CI Lower": roc_auc_ci["lower_95"],
        "CI Upper": roc_auc_ci["upper_95"]
    },
    {
        "Metric": "PR-AUC",
        "Estimate": pr_auc_ci["estimate"],
        "CI Lower": pr_auc_ci["lower_95"],
        "CI Upper": pr_auc_ci["upper_95"]
    },
    {
        "Metric": "Brier",
        "Estimate": brier_ci["estimate"],
        "CI Lower": brier_ci["lower_95"],
        "CI Upper": brier_ci["upper_95"]
    }
])

display(canonical_bootstrap_summary)

In [ ]:
import json

with open(CONFIG_PATH, "r") as f:
    config_on_disk = json.load(f)

print(json.dumps(config_on_disk, indent=4))

# Verify the canonical performance values
assert config_on_disk["performance"]["test_roc_auc"] == roc_auc_score(
    y_test, test_probabilities
)

assert config_on_disk["performance"]["test_pr_auc"] == average_precision_score(
    y_test, test_probabilities
)

assert config_on_disk["performance"]["test_brier"] == brier_score_loss(
    y_test, test_probabilities
)

# Verify the bootstrap CIs
assert config_on_disk["bootstrap_95_ci"]["roc_auc"] == [
    roc_auc_ci["lower_95"],
    roc_auc_ci["upper_95"]
]

assert config_on_disk["bootstrap_95_ci"]["pr_auc"] == [
    pr_auc_ci["lower_95"],
    pr_auc_ci["upper_95"]
]

assert config_on_disk["bootstrap_95_ci"]["brier"] == [
    brier_ci["lower_95"],
    brier_ci["upper_95"]
]

print("\n✅ model_config.json matches the current canonical evaluation results.")

## Deployment packaging

From here, the original development notebook went on to regenerate
`src/credit_risk/preprocessing.py`, `src/credit_risk/predictor.py`, and
`app/main.py` by writing strings to disk, then mounted Google Drive and
zipped the project for download — all Colab session mechanics.

Those modules are maintained directly as real, version-controlled files in
this repository rather than being regenerated here:

- [`src/credit_risk/preprocessing.py`](../src/credit_risk/preprocessing.py)
- [`src/credit_risk/predictor.py`](../src/credit_risk/predictor.py)
- [`app/main.py`](../app/main.py)

They implement the same `prepare_tree_features` logic and the same
SHAP-based reason-code generation developed and verified in Sections 10 and
15 above, wired into a FastAPI service instead of a notebook cell.
